# SpaceX Falcon 9 - Interactive Visual Analytics with Folium
This notebook maps Falcon 9 launch sites, overlays launch outcomes, and measures proximity to surrounding infrastructure. The purpose is to understand the geographic context of each site.


In [ ]:
import pandas as pd
import folium
from folium.plugins import MarkerCluster
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

url = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv'
df = pd.read_csv(url)
site_df = df[['LaunchSite','Latitude','Longitude']].drop_duplicates()
site_df


## Launch-site markers
The three launch sites in the cleaned 90-flight dataset are CCAFS SLC 40 and KSC LC 39A on Florida's Space Coast, plus VAFB SLC 4E on California's coast.


In [ ]:
center = [site_df['Latitude'].mean(), site_df['Longitude'].mean()]
site_map = folium.Map(location=center, zoom_start=4)
for _, row in site_df.iterrows():
    folium.CircleMarker(
        [row.Latitude,row.Longitude], radius=8,
        color='#2563eb', fill=True, fill_opacity=.85,
        popup=row.LaunchSite
    ).add_to(site_map)
site_map


## Cluster individual launch records by outcome
Green markers represent successful first-stage outcomes (`Class=1`) and red markers represent unsuccessful/non-success outcomes (`Class=0`). MarkerCluster keeps overlapping launch records readable.


In [ ]:
cluster_map = folium.Map(location=[29.5,-92], zoom_start=4)
cluster = MarkerCluster().add_to(cluster_map)
for _, row in df.iterrows():
    color = 'green' if row.Class == 1 else 'red'
    folium.Marker(
        [row.Latitude,row.Longitude],
        popup=f"Flight {row.FlightNumber} | {row.LaunchSite} | {'Success' if row.Class else 'No success'}",
        icon=folium.Icon(color=color)
    ).add_to(cluster)
cluster_map


## Proximity analysis
The Florida launch sites are near the coastline and transport infrastructure, while VAFB is positioned on the Pacific coast. Coastal placement supports range safety: launch trajectories can travel over ocean rather than populated areas.


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1,p2 = radians(lat1),radians(lat2)
    dlat,dlon = radians(lat2-lat1),radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(p1)*cos(p2)*sin(dlon/2)**2
    return 2*R*atan2(sqrt(a),sqrt(1-a))

# Example: annotate known site coordinates and use the same function for nearby
# coastline/city/railway/highway points identified in the Folium lab.
for _, r in site_df.iterrows():
    print(r.LaunchSite, round(r.Latitude,4), round(r.Longitude,4))
